# HW3 · Ноутбук 2b (GPU T4): сильніший енкодер (v3, більше даних)
Акселератор: **GPU T4**, **Internet On**. Запускай через *Save Version → Save & Run All*.

Спершу перезапусти ноутбук 0 (train тепер ~100k, валідація та сама). Час: **mDeBERTa ~4-4.5 год, XLM-R ~2-2.5 год** (2 епохи).

**Запусти двічі** (Copy & Edit): з `MODEL = "microsoft/mdeberta-v3-base"` і з `MODEL = "xlm-roberta-base"`. Їх можна запускати одночасно у двох сесіях.

У назві файлів є розмір train (`..._dom_99k`), тож результати не плутаються з ранішими прогонами на 32k.

## Шапка (однакова в усіх ноутбуках)
Читає дані конкурсу та `split.csv` з ноутбука 0. Перед запуском: *Add Input → Notebook Output Files* і додай outputs ноутбука 0. Спліт групований, тож валідація чесно імітує тест (там теж нових текстів не було в train).

In [1]:
import os, glob, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.metrics import f1_score, classification_report, confusion_matrix

ID, TEXT, LABEL = "id", "text", "label"
SEED = 42
LABELS = ["human", "ai", "fusion"]
OUT = "/kaggle/working"

def find(name):
    return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
ss = pd.read_csv(find("sample_submission.csv"))

# спліт береться з ноутбука 0 (split.csv): вся група "людський текст + його AI/fusion версії" лежить в одній частині
sp = pd.read_csv(find("split.csv"))
d = train.merge(sp, on=ID)
tr = d[d.part == "tr"].reset_index(drop=True)
va = d[d.part == "va"].reset_index(drop=True)
l2i = {l: i for i, l in enumerate(LABELS)}
ytr, yva = tr[LABEL].map(l2i).values, va[LABEL].map(l2i).values
print(train.columns.tolist(), "| train:", len(tr), "val:", len(va), "test:", len(test))

def save_sub(p, fn):  # p = матриця ймовірностей для test, пише файл у форматі Kaggle
    out = pd.DataFrame({ID: test[ID], LABEL: np.array(LABELS)[p.argmax(1)]})
    out.set_index(ID).loc[ss[ID]].reset_index().to_csv(f"{OUT}/{fn}", index=False)

['id', 'text', 'lang', 'domain', 'label'] | train: 100002 val: 8005 test: 25000


## Налаштування і токенізація
Довгий текст скорочуємо до 512 токенів: половина з початку, половина з кінця (`head_tail`). Для fusion кінець тексту може відрізнятися від початку.

In [2]:
import torch, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL = "xlm-roberta-base"        # другий прогін: "microsoft/mdeberta-v3-base"
MAX_LEN = 512
EPOCHS, BS, ACC, LR = 2, 8, 2, 2e-5   # справжній батч = BS*ACC = 16
dev = "cuda"

tok = AutoTokenizer.from_pretrained(MODEL)
tok.model_max_length = 10**6

def prep(d):  # дописуємо домен і мову на початок тексту
    return ("[" + d["domain"] + " | " + d["lang"] + "] " + d[TEXT].str.slice(0, 20000)).tolist()

def encode(texts):
    k, res = MAX_LEN - 2, []
    for s in range(0, len(texts), 10000):      # по 10k текстів, щоб не з'їсти всю пам'ять
        for ids in tok(texts[s:s + 10000], add_special_tokens=False)["input_ids"]:
            if len(ids) > k:
                ids = ids[:k // 2] + ids[-(k - k // 2):]
            res.append([tok.cls_token_id] + ids + [tok.sep_token_id])
    return res

TAG = f"{MODEL.split('/')[-1]}_ht_{MAX_LEN}_dom_{len(tr)//1000}k"   # у назві розмір train, щоб не плутати з ранішими
tr_ids, va_ids, te_ids = encode(prep(tr)), encode(prep(va)), encode(prep(test))
print(round(np.mean([len(x) == MAX_LEN for x in tr_ids]), 3), "частка обрізаних текстів")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

0.65 частка обрізаних текстів


## Батчі й предікт
`collate` вирівнює тексти в батчі до однакової довжини (padding). `predict` сортує тексти за довжиною, щоб менше витрачати на padding, і повертає ймовірності.

In [3]:
from torch.utils.data import DataLoader

def collate(b):
    m = max(len(x[0]) for x in b)
    inp = torch.full((len(b), m), tok.pad_token_id)
    att = torch.zeros((len(b), m), dtype=torch.long)
    for k, (ids, _) in enumerate(b):
        inp[k, :len(ids)] = torch.tensor(ids); att[k, :len(ids)] = 1
    return inp, att, torch.tensor([x[1] for x in b])

@torch.no_grad()
def predict(ids, bs=32):
    model.eval()
    order = np.argsort([len(i) for i in ids])
    out = np.zeros((len(ids), 3), dtype=np.float32)
    for s in range(0, len(ids), bs):
        idx = order[s:s + bs]
        x, a, _ = collate([(ids[j], 0) for j in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(input_ids=x.to(dev), attention_mask=a.to(dev)).logits
        out[idx] = torch.softmax(lg.float(), -1).cpu().numpy()
    return out

## Навчання
Дотренування (fine-tuning) у fp16 з накопиченням градієнта (2 мікробатчі по 8 = батч 16, щоб влізло в пам'ять T4). Після кожної епохи рахуємо macro-F1 на валідації й, якщо стало краще, зберігаємо ймовірності та сабміт.

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3).float().to(dev)   # .float(): mDeBERTa інакше вантажиться в fp16
dl = DataLoader(list(zip(tr_ids, ytr.tolist())), batch_size=BS, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = EPOCHS * (len(dl) // ACC)
sch = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
scaler = torch.cuda.amp.GradScaler()

best_f, t0 = -1, time.time()
for ep in range(EPOCHS):
    model.train(); opt.zero_grad()
    for step, (x, a, y) in enumerate(dl):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_ids=x.to(dev), attention_mask=a.to(dev), labels=y.to(dev)).loss / ACC
        scaler.scale(loss).backward()
        if (step + 1) % ACC == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
        if step % 1000 == 0:
            print(f"ep{ep} step{step}/{len(dl)} loss={loss.item() * ACC:.3f} t={(time.time()-t0)/60:.1f}хв")
    pva = predict(va_ids)
    f = f1_score(yva, pva.argmax(1), average="macro")
    print(f"=== епоха {ep}: val macro-F1 = {f:.4f}")
    if f > best_f:
        best_f, best_pva = f, pva
        np.save(f"{OUT}/enc_{TAG}_val.npy", pva)
        pte = predict(te_ids)
        np.save(f"{OUT}/enc_{TAG}_test.npy", pte)
        save_sub(pte, f"submission_enc_{TAG}.csv")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ep0 step0/12501 loss=1.105 t=0.0хв
ep0 step1000/12501 loss=0.835 t=4.7хв
ep0 step2000/12501 loss=1.056 t=9.5хв
ep0 step3000/12501 loss=0.833 t=14.3хв
ep0 step4000/12501 loss=0.176 t=19.1хв
ep0 step5000/12501 loss=0.086 t=23.9хв
ep0 step6000/12501 loss=0.088 t=28.7хв
ep0 step7000/12501 loss=0.081 t=33.5хв
ep0 step8000/12501 loss=0.135 t=38.3хв
ep0 step9000/12501 loss=0.019 t=43.0хв
ep0 step10000/12501 loss=0.149 t=47.8хв
ep0 step11000/12501 loss=0.009 t=52.6хв
ep0 step12000/12501 loss=0.601 t=57.3хв
=== епоха 0: val macro-F1 = 0.9170
ep1 step0/12501 loss=0.024 t=63.6хв
ep1 step1000/12501 loss=0.121 t=68.3хв
ep1 step2000/12501 loss=0.003 t=73.1хв
ep1 step3000/12501 loss=0.088 t=77.8хв
ep1 step4000/12501 loss=0.004 t=82.6хв
ep1 step5000/12501 loss=0.057 t=87.3хв
ep1 step6000/12501 loss=0.848 t=92.0хв
ep1 step7000/12501 loss=0.002 t=96.7хв
ep1 step8000/12501 loss=0.630 t=101.4хв
ep1 step9000/12501 loss=0.007 t=106.1хв
ep1 step10000/12501 loss=0.015 t=110.8хв
ep1 step11000/12501 loss=0.022 

## Звіт по класах і по доменах
Таблиця по доменах покаже, де модель слабша (для звіту й захисту).

In [5]:
print(TAG, "best macro-F1:", round(best_f, 4))
print(classification_report(yva, best_pva.argmax(1), target_names=LABELS, digits=3))
res = pd.DataFrame({"domain": va["domain"], "ok": best_pva.argmax(1) == yva})
print(res.groupby("domain")["ok"].mean().round(3))

xlm-roberta-base_ht_512_dom_100k best macro-F1: 0.9221
              precision    recall  f1-score   support

       human      0.898     0.842     0.869      1950
          ai      0.992     0.997     0.995      3556
      fusion      0.885     0.922     0.903      2499

    accuracy                          0.936      8005
   macro avg      0.925     0.920     0.922      8005
weighted avg      0.936     0.936     0.935      8005

domain
arxiv_papers              0.957
blog_authorship_corpus    0.880
bundestag                 0.924
cnn_news                  0.969
euro_court_cases          0.869
gutenberg                 0.945
house_of_commons          0.967
student_essays            0.970
Name: ok, dtype: float64
